<a href="https://colab.research.google.com/github/ibnu-aulia/arabic-text-analyzer/blob/main/arabert_restaurant_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning AraBERT on Arabic Restaurant Reviews

Part of my Arabic NLP learning path. This notebook continues the restaurant sentiment project: I fine-tune AraBERT (`aubmindlab/bert-base-arabertv02`) on the same data and the same train/test split as my TF-IDF + Logistic Regression models, then compare the results and read the errors.

**Setup:** use a GPU runtime (Runtime > Change runtime type > T4 GPU). Training uses `fp16=True`, which needs a GPU.

**Sections**
1. Setup
2. Attention in numbers (toy example)
3. A ready-made Arabic sentiment pipeline
4. Data and the same split as before
5. AraBERT preprocessing and tokenizer
6. Datasets and model
7. Fine-tuning
8. Evaluation
9. Error analysis
10. Summary and limitations

## 1. Setup

In [1]:
!pip install arabert -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Attention in numbers (toy example)

Attention turns relevance scores into weights that sum to 1 (softmax), then mixes the word values by those weights. The scores and values below are made up for practice, not taken from a real model.

Expected: the first word gets the biggest weight (about 0.665), the weights sum to 1.0, and the output is about 14.25, closer to 10 than to 30.

In [2]:
import math

words = ["ما", "انصح", "المطعم"]
scores = [2,1,0]
values = [10,20,30]

exps = [math.exp(s) for s in scores]
total = sum(exps)
weights = [e / total for e in exps]
output = sum(w * v for w, v in zip(weights, values))

print(weights)
print(sum(weights))
print(output)

[0.6652409557748219, 0.24472847105479764, 0.09003057317038046]
1.0
14.247896173955585


## 3. A ready-made Arabic sentiment pipeline

Before training anything, I tried a sentiment model that is already fine-tuned (`CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment`) on four sentences. The negated sentence `ما انصح بهذا المطعم` ("I don't recommend this restaurant") came out as neutral with a low score (0.611), while the other three were around 0.98. Four sentences prove nothing, so this is only a first look.

In [3]:
from transformers import pipeline

classifier = pipeline("text-classification", model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment")

sentences = [ "الأكل لذيذ جدا", "الخدمة سيئة", "ما انصح بهذا المطعم", "مرة حلو", ]

results = classifier(sentences)

for s,r in zip(sentences, results):
    print(s, "->", r["label"], round(r["score"], 3))

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

الأكل لذيذ جدا -> positive 0.978
الخدمة سيئة -> negative 0.986
ما انصح بهذا المطعم -> neutral 0.611
مرة حلو -> positive 0.988


## 4. Data and the same split as before

Same dataset (`hadyelsahar/ar_res_reviews`), same cleaning, and the same split as the TF-IDF models (`test_size=0.2`, `random_state=42`, `stratify=y`), so the comparison is fair. The cleaned column is only used to drop empty and duplicate reviews. AraBERT gets the original text.

Expected: 8364 rows, then 8237 rows after cleaning (5850 positive, 2387 negative), and a split of 6589 train and 1648 test (1170 positive, 478 negative).

In [4]:
import re
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

ds = load_dataset("hadyelsahar/ar_res_reviews")
df = ds["train"].to_pandas()
print(df.shape)

def clean(text):
    text = re.sub(r"[أإآ]", "ا", text)
    text = text.replace("ى", "ي")
    text = re.sub(r"[^\u0621-\u064A\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean"] = df["text"].apply(clean)
df = df[df["clean"] != ""]
df = df.drop_duplicates(subset="clean")
print(df.shape)
print(df["polarity"].value_counts())

X = df["text"]
y = df["polarity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(len(X_train), len(X_test))
print(y_test.value_counts())

README.md:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.89MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8364 [00:00<?, ? examples/s]

(8364, 4)
(8237, 5)
polarity
1    5850
0    2387
Name: count, dtype: int64
6589 1648
polarity
1    1170
0     478
Name: count, dtype: int64


## 5. AraBERT preprocessing and tokenizer

`arabertv02` is used with `ArabertPreprocessor` (no Farasa needed). The model, tokenizer, and preprocessing must match, otherwise the results break without any error. Reviews are cut at 128 tokens, so I check how many get truncated.

In [5]:
from arabert.preprocess import ArabertPreprocessor
from transformers import AutoTokenizer

model_name = "aubmindlab/bert-base-arabertv02"
prep = ArabertPreprocessor(model_name=model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_texts = [prep.preprocess(t) for t in X_train]
test_texts = [prep.preprocess(t) for t in X_test]
print(len(train_texts), len(test_texts))

print(list(X_train)[0])
print(train_texts[0])

enc = tokenizer(train_texts, truncation=True, max_length=128)
lengths = [len(ids) for ids in enc["input_ids"]]
print(max(lengths))
print(sum(1 for l in lengths if l == 128))
print(round(sum(1 for l in lengths if l == 128) / len(lengths) * 100, 1))

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

6589 1648
الحمد لله تم افتتاح فرع شاطر شارع الاربعين  وصار قريب لي  كنت اول اضرب مشوار لسوق الشاطيء لعيونهم  انصحكم بتجربة الشاورما لانها مميزه
الحمد لله تم افتتاح فرع شاطر شارع الاربعين وصار قريب لي كنت اول اضرب مشوار لسوق الشاطيء لعيونهم انصحكم بتجربة الشاورما لانها مميزه
128
678
10.3


## 6. Datasets and model

The data is wrapped in a small Dataset class, and AraBERT is loaded with a new 2-class head. The warning about `classifier.weight` and `classifier.bias` being newly initialized is expected: the head is new and gets trained. The `cls.predictions...` keys are leftovers from pretraining and are not used.

In [6]:
import torch
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

train_enc = tokenizer(train_texts, truncation=True, max_length=128)
test_enc = tokenizer(test_texts, truncation=True, max_length=128)

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item["labels"] = int(self.labels[i])
        return item

train_ds = ReviewDataset(train_enc, y_train)
test_ds = ReviewDataset(test_enc, y_test)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
collator = DataCollatorWithPadding(tokenizer)

print(len(train_ds), len(test_ds))
print(train_ds[0]["labels"], len(train_ds[0]["input_ids"]))
print(sum(p.numel() for p in model.parameters()))

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


6589 1648
1 33
135194882


## 7. Fine-tuning

3 epochs, batch size 16, learning rate 2e-5, seed 42, no class weighting. Validation was run each epoch for monitoring only. I report the final model after epoch 3, not the best epoch.

Validation loss went up from epoch 1 to epoch 3 while training loss went down, which looks like the start of overfitting.

In [7]:
import numpy as np
from transformers import TrainingArguments, Trainer

print(torch.cuda.is_available())

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

args = TrainingArguments(
    output_dir="arabert_restaurant",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

True


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.280346,0.896238
2,0.337705,0.331667,0.898665
3,0.202748,0.369516,0.895024


TrainOutput(global_step=1236, training_loss=0.2502128915879333, metrics={'train_runtime': 156.007, 'train_samples_per_second': 126.706, 'train_steps_per_second': 7.923, 'total_flos': 1255763289469440.0, 'train_loss': 0.2502128915879333, 'epoch': 3.0})

## 8. Evaluation on the same 1648 test reviews

In [8]:
from sklearn.metrics import classification_report

pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids

print(classification_report(y_true, y_pred, target_names=["negative", "positive"], digits=2))

              precision    recall  f1-score   support

    negative       0.85      0.78      0.81       478
    positive       0.91      0.94      0.93      1170

    accuracy                           0.90      1648
   macro avg       0.88      0.86      0.87      1648
weighted avg       0.89      0.90      0.89      1648



Comparison with the best TF-IDF model from my earlier notebook (`arabic_review_sentiment.ipynb`). AraBERT numbers are ranges over my runs of this notebook, because results shift slightly between runs even with a fixed seed:

| Metric | TF-IDF + LR (balanced, 20k uni+bigram) | AraBERT (range over runs) |
|---|---|---|
| Accuracy | 0.86 | 0.89 |
| Precision (negative) | 0.74 | 0.83 to 0.84 |
| Recall (negative) | 0.77 | 0.76 to 0.78 |
| F1 (negative) | 0.76 | 0.80 to 0.81 |
| Macro F1 | 0.83 | 0.86 |

AraBERT is better on this split, mostly through precision on negative reviews. Recall on negative reviews is about the same as the TF-IDF model.

## 9. Error analysis

Negative reviews that AraBERT predicted as positive: between 103 and 113 in a few runs. The exact reviews shown change a little between runs. I read the first 8 of them.

In [9]:
test_df = pd.DataFrame({"text": list(X_test), "true": y_true, "pred": y_pred})
missed = test_df[(test_df["true"] == 0) & (test_df["pred"] == 1)]

print(len(missed))
print(missed["text"].str.contains("ما انصح").sum())

for t in missed["text"].head(8):
    print(t[:150])
    print("---")

107
0
اسعاره رخيصه والخدمه   منوسطه والنظافه ممتازه الطعم عادي خاصه باالاكل الهندي فيه اللي افضل منه
---
السلام عليكم 
 تقييمي الشخصي :: 
 الاكل : جيد جدا بس بارد شوي .. والكمية في الصحون قليلة جدا  
 الخدمة : الشايب الي يستقبلك كنه يكد عليك .. الباقين حل
---
هو اكلهم حلو وماعليه  
 بس مافيه نظافة بالاكل  
 مره حصلت حشرات بالاكل
---
خصوصا الباربيكيو بيتزا
---
المطعم هذا كان ممتاز لكن بعد ما فتح مطعم عالخفيف غطى على ابوووووه  
 عالخفيف ارخص بكثير والنظافة ممتازة والأكل لذيذ خصوصاً مناقيش الجبن 
  
 تحياتي
---
الجو العام للمطعم متوسط 
 مشاوي وفطائر وشاروما .. الخ  
 يوجد قسم للحفلات الخارجية 
 الخدمة سريعه  
 مذاق الأكل ماعجبني صراحه 
 الاسعار في متناول الجم
---
الاكل بشكل عام جيد، الفطائر تحديداً ممتازة. 
 المشكلة تبقى في الخدمة والسرعة، الترحيب او الابتسام ممنوع على العاملين، وعادة لايحترمون الاوقت فهم دائما
---
للمطعم مكان بارز على تقاطع شارع العليا العام و طريق الملك عبدالله.  
  
 ذهبت مرة وحدي لزيارته. انتظرت في الواجهة فترة ليتم اجلاسي (رغم وجود 8 طاولات 
---


From the 8 examples I read (a small sample, so no percentages):

- **Mixed reviews:** praise first, complaint later (cheap and clean but average service, good food but cold and small portions, good food but rude staff).
- **Comparison with another place**, where the negative is implied (`جافا تايم أفضل منه`).
- **Negation:** `ماعجبني` and `لم يعجبني` (negation attached to the verb) and `ما نجحت` appeared in some of the missed reviews. This is different from the separated `ما انصح`. That exact phrase did not appear in any of the missed reviews, but the search only matches that exact spelling.
- **Unclear label:** one review is only a short fragment.

## 10. Summary and limitations

**What I did:** fine-tuned AraBERT on 6589 restaurant reviews and evaluated it on 1648 test reviews, using the same split as my TF-IDF models.

**Result:** accuracy 0.89 (vs 0.86), F1 for negative reviews 0.80 to 0.81 (vs 0.76), recall for negative reviews 0.76 to 0.78 (vs 0.77), depending on the run.

**Limitations**
- some runs, one split, no cross-validation. The runs differed by up to 0.02 on negative recall, so small gaps should not be over-read.
- The TF-IDF model used `class_weight="balanced"` and AraBERT did not, so the comparison is not perfectly equal.
- About 10.3% of training reviews were truncated at 128 tokens.
- Validation loss rose during training (possible overfitting).
- The error analysis covers only 8 examples.
- The trained model was not saved.

**Next:** try class weights, check negation and dialect cases on purpose, and run cross-validation.